TEXT MINING FINAL PROJECT


1. We import libraries and read the csv

In [1]:
import os
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

from pprint import pprint

from time import time

from sklearn.feature_extraction.text import TfidfTransformer, CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import precision_recall_curve, average_precision_score

# path to where you saved the training dataset that I provided
readin = ''

spitout= ''

# this is loading training dataset
filename ="reviews.csv"

# loading the training data with error handling
try:
    data_train = pd.read_csv(os.path.join(readin, filename), sep=',', encoding='utf-8')
    data_train.info()
except FileNotFoundError:
    print(f"Error: The file {filename} was not found in the directory {readin}.")
    data_train = pd.DataFrame()  # Create an empty DataFrame to avoid stopping the process
except Exception as e:
    print(f"An error occurred while loading the data: {e}")
    data_train = pd.DataFrame()  # Create an empty DataFrame to avoid stopping the process

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1019270 entries, 0 to 1019269
Data columns (total 6 columns):
 #   Column         Non-Null Count    Dtype 
---  ------         --------------    ----- 
 0   listing_id     1019270 non-null  int64 
 1   id             1019270 non-null  int64 
 2   date           1019270 non-null  object
 3   reviewer_id    1019270 non-null  int64 
 4   reviewer_name  1019266 non-null  object
 5   comments       1019160 non-null  object
dtypes: int64(3), object(3)
memory usage: 46.7+ MB


In [2]:
comments = data_train['comments']

comments.head()

0    Excelente lugar y buena ubicación, repetiría e...
1              Very good host and always ready to help
2    Excellent place to stay and great location. Re...
3    Easy and quick communication with the host, gr...
4    Beautiful apartment in a very great neighbourh...
Name: comments, dtype: object

In [3]:
#!pip install deep-translator tqdm

Translate the reviews using Google Translator, partitioning the info in text files. This took almost one week.

In [4]:
import json
import pandas as pd
from deep_translator import GoogleTranslator
from tqdm import tqdm
import os
import ctypes

# Prevent the computer from sleeping
ES_CONTINUOUS = 0x80000000
ES_SYSTEM_REQUIRED = 0x00000001
ctypes.windll.kernel32.SetThreadExecutionState(ES_CONTINUOUS | ES_SYSTEM_REQUIRED)

# crear traductor
translator = GoogleTranslator(source='auto', target='en')

# activar barra de progreso
tqdm.pandas()

# función segura
def translate_text(text):
    try:
        if pd.isna(text) or str(text).strip() == "":
            return text
        return translator.translate(text)
    except Exception:
        return text

# --- Checkpoint helpers (JSON-based, with legacy line-based fallback) ---
def save_checkpoint(filepath, values):
    """Save a batch as a JSON list so embedded newlines are preserved."""
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump([str(v) for v in values], f, ensure_ascii=False)

def load_checkpoint(filepath, expected_size):
    """Load a checkpoint, trying JSON first then falling back to line-based."""
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()
    # Try JSON first
    try:
        data = json.loads(content)
        if isinstance(data, list):
            return data[:expected_size]
    except (json.JSONDecodeError, ValueError):
        pass
    # Legacy line-based format — join excess lines caused by embedded \n
    lines = content.split("\n")
    if lines and lines[-1] == "":
        lines = lines[:-1]
    if len(lines) == expected_size:
        return lines
    # Mismatch: re-save as JSON so future loads are clean
    print(f"  ⚠ {filepath}: got {len(lines)} lines, expected {expected_size}; truncating/padding & converting to JSON")
    if len(lines) > expected_size:
        result = lines[:expected_size]
    else:
        result = lines + [""] * (expected_size - len(lines))
    save_checkpoint(filepath, result)
    return result

# --- Batch translation with checkpoint saves every 1,000 rows ---
batch_size = 1_000
total = len(comments)
checkpoint_dir = "translation_checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

# Find already completed batches to allow resuming
completed_batches = set()
for f in os.listdir(checkpoint_dir):
    if f.startswith("batch_") and f.endswith(".txt"):
        try:
            idx = int(f.replace("batch_", "").replace(".txt", ""))
            completed_batches.add(idx)
        except ValueError:
            pass

comments_en = pd.Series([""] * total, index=comments.index)

for start in tqdm(range(0, total, batch_size), desc="Batches"):
    end = min(start + batch_size, total)
    batch_file = os.path.join(checkpoint_dir, f"batch_{start}.txt")

    if start in completed_batches:
        # Load from checkpoint
        loaded = load_checkpoint(batch_file, end - start)
        comments_en.iloc[start:end] = loaded
        print(f"Loaded checkpoint: rows {start}-{end-1}")
    else:
        # Translate this batch
        batch = comments.iloc[start:end]
        translated_batch = batch.progress_apply(translate_text)
        comments_en.iloc[start:end] = translated_batch.values

        # Save checkpoint (JSON format)
        save_checkpoint(batch_file, translated_batch.values)
        print(f"Saved checkpoint: rows {start}-{end-1} -> {batch_file}")

print(f"\nTranslation complete. {total} rows processed.")

# Allow the computer to sleep again after the process is complete
ctypes.windll.kernel32.SetThreadExecutionState(ES_CONTINUOUS)

Batches:  11%|█         | 109/1020 [00:00<00:01, 544.67it/s]

Loaded checkpoint: rows 0-999
Loaded checkpoint: rows 1000-1999
Loaded checkpoint: rows 2000-2999
Loaded checkpoint: rows 3000-3999
Loaded checkpoint: rows 4000-4999
Loaded checkpoint: rows 5000-5999
Loaded checkpoint: rows 6000-6999
Loaded checkpoint: rows 7000-7999
Loaded checkpoint: rows 8000-8999
Loaded checkpoint: rows 9000-9999
Loaded checkpoint: rows 10000-10999
Loaded checkpoint: rows 11000-11999
Loaded checkpoint: rows 12000-12999
Loaded checkpoint: rows 13000-13999
Loaded checkpoint: rows 14000-14999
Loaded checkpoint: rows 15000-15999
Loaded checkpoint: rows 16000-16999
Loaded checkpoint: rows 17000-17999
Loaded checkpoint: rows 18000-18999
Loaded checkpoint: rows 19000-19999
Loaded checkpoint: rows 20000-20999
Loaded checkpoint: rows 21000-21999
Loaded checkpoint: rows 22000-22999
Loaded checkpoint: rows 23000-23999
Loaded checkpoint: rows 24000-24999
Loaded checkpoint: rows 25000-25999
Loaded checkpoint: rows 26000-26999
Loaded checkpoint: rows 27000-27999
Loaded checkpoin

Batches:  17%|█▋        | 171/1020 [00:00<00:01, 544.14it/s]

Loaded checkpoint: rows 109000-109999
Loaded checkpoint: rows 110000-110999
Loaded checkpoint: rows 111000-111999
Loaded checkpoint: rows 112000-112999
Loaded checkpoint: rows 113000-113999
Loaded checkpoint: rows 114000-114999
Loaded checkpoint: rows 115000-115999
Loaded checkpoint: rows 116000-116999
Loaded checkpoint: rows 117000-117999
Loaded checkpoint: rows 118000-118999
Loaded checkpoint: rows 119000-119999
Loaded checkpoint: rows 120000-120999
Loaded checkpoint: rows 121000-121999
Loaded checkpoint: rows 122000-122999
Loaded checkpoint: rows 123000-123999
Loaded checkpoint: rows 124000-124999
Loaded checkpoint: rows 125000-125999
Loaded checkpoint: rows 126000-126999
Loaded checkpoint: rows 127000-127999
Loaded checkpoint: rows 128000-128999
Loaded checkpoint: rows 129000-129999
Loaded checkpoint: rows 130000-130999
Loaded checkpoint: rows 131000-131999
Loaded checkpoint: rows 132000-132999
Loaded checkpoint: rows 133000-133999
Loaded checkpoint: rows 134000-134999
Loaded check

Batches:  28%|██▊       | 281/1020 [00:00<00:01, 499.68it/s]

...

Batches:  88%|████████▊ | 901/1020 [00:01<00:00, 646.48it/s]

Loaded checkpoint: rows 824000-824999
Loaded checkpoint: rows 825000-825999
Loaded checkpoint: rows 826000-826999
Loaded checkpoint: rows 827000-827999
Loaded checkpoint: rows 828000-828999
Loaded checkpoint: rows 829000-829999
Loaded checkpoint: rows 830000-830999
Loaded checkpoint: rows 831000-831999
Loaded checkpoint: rows 832000-832999
Loaded checkpoint: rows 833000-833999
Loaded checkpoint: rows 834000-834999
Loaded checkpoint: rows 835000-835999
Loaded checkpoint: rows 836000-836999
Loaded checkpoint: rows 837000-837999
Loaded checkpoint: rows 838000-838999
Loaded checkpoint: rows 839000-839999
Loaded checkpoint: rows 840000-840999
Loaded checkpoint: rows 841000-841999
Loaded checkpoint: rows 842000-842999
Loaded checkpoint: rows 843000-843999
Loaded checkpoint: rows 844000-844999
Loaded checkpoint: rows 845000-845999
Loaded checkpoint: rows 846000-846999
Loaded checkpoint: rows 847000-847999
Loaded checkpoint: rows 848000-848999
Loaded checkpoint: rows 849000-849999
Loaded check

Batches: 100%|██████████| 1020/1020 [00:01<00:00, 565.91it/s]

Loaded checkpoint: rows 954000-954999
Loaded checkpoint: rows 955000-955999
Loaded checkpoint: rows 956000-956999
Loaded checkpoint: rows 957000-957999
Loaded checkpoint: rows 958000-958999
Loaded checkpoint: rows 959000-959999
Loaded checkpoint: rows 960000-960999
Loaded checkpoint: rows 961000-961999
Loaded checkpoint: rows 962000-962999
Loaded checkpoint: rows 963000-963999
Loaded checkpoint: rows 964000-964999
Loaded checkpoint: rows 965000-965999
Loaded checkpoint: rows 966000-966999
Loaded checkpoint: rows 967000-967999
Loaded checkpoint: rows 968000-968999
Loaded checkpoint: rows 969000-969999
Loaded checkpoint: rows 970000-970999
Loaded checkpoint: rows 971000-971999
Loaded checkpoint: rows 972000-972999
Loaded checkpoint: rows 973000-973999
Loaded checkpoint: rows 974000-974999
Loaded checkpoint: rows 975000-975999
Loaded checkpoint: rows 976000-976999
Loaded checkpoint: rows 977000-977999
Loaded checkpoint: rows 978000-978999
Loaded checkpoint: rows 979000-979999
Loaded check

-2147483647

In [5]:
# Create reviews_translated.csv with all original columns + new "Reviews_Translation" column
data_translated = data_train.copy()
data_translated["Reviews_Translation"] = comments_en.values
data_translated.to_csv("reviews_translated.csv", index=False, encoding="utf-8")
print(f"Saved reviews_translated.csv with {len(data_translated)} rows and columns: {list(data_translated.columns)}")

Saved reviews_translated.csv with 1019270 rows and columns: ['listing_id', 'id', 'date', 'reviewer_id', 'reviewer_name', 'comments', 'Reviews_Translation']
